# Figure 6 Preparation

In [ ]:
# to create pkl files with angles for certain type 
# run i.e.
# python tools/calculate_angles_facerec.py mean_bias --data_names_file="fig_6.txt"
# available types are:
#     'mean_bias',
#     'only_mean',
#     'only_bias',
#     'nothing'

In [3]:
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")

import pickle
import numpy as np

In [ ]:
s = f"""
| FaceRec | After Conv | After Shifting | $\\Delta$ After Conv | $\\Delta$ After Shifting |
| -- | -- | -- | -- | -- |
"""

layer_names = [
    'After_conv',
    'After_shifting',
    'Before_conv_vs_After_conv',
    'Before_shifting_vs_After_shifting',
]

def compute_avg(data):
    res = []
    for i in range(3):
        vals = [layer['mean'][0][i] 
                for layer in data.values() 
                if not np.isnan(layer['mean'][0][i])]
        res.append(np.mean(vals))
        vals = [np.sqrt(layer['var'][0][i]) 
                for layer in data.values() 
                if not np.isnan(layer['var'][0][i])]
        res.append(np.mean(vals))
    return res


def format_metrics(avg_metrics):
    labels = ['in', 'out', 'back']
    parts = []

    for i, label in enumerate(labels):
        mean = avg_metrics[2 * i]
        std = avg_metrics[2 * i + 1]
        parts.append(f'{label} {mean:.2f}±{std:.2f}<br>')

    return ''.join(parts)


for type_ in [
    'mean_bias',
    'only_mean',
    'only_bias',
    'nothing'
]:
    PATH = f'heap/arcface/lfw/r50_{type_}/'
    cells = []
    for layer_data_name in layer_names:
        with open(PATH + layer_data_name + '.pkl', 'rb') as fr:
            res = pickle.load(fr)
        avg_metrics = compute_avg(res)
        cells.append(format_metrics(avg_metrics))

    s += f'''| {type_} | {' | '.join(cells)} |\n'''

print(s)

## Figure 8 Preparation

In [42]:
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")

import pickle
import numpy as np

from collections import defaultdict

In [ ]:
# to create pkl files with angles for certain type 
# run i.e.
# python tools/calculate_angles_facerec.py mean_bias --data_names_file="fig_8.txt"
# available types are:
#     'mean_bias',
#     'only_mean',
#     'only_bias',
#     'nothing'

In [ ]:
PATH = f'heap/arcface/lfw/r50_mean_bias/'

def compute_avg(data):
    res = []
    vals = [np.nanmean(layer['mean'][0])
            for layer in data.values() 
            ]
    res.append(np.mean(vals))
    vals = [np.sqrt(layer['var'][0]) 
            for layer in data.values() 
           ]
    res.append(np.mean(vals))
    return res


VS = [0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 0.9, 0.95]
RES = defaultdict(list)
for v in VS:
    for t in ['top', 'low']:
        for s in ['center', 's']:
            CORR_TYPE = f'{t}_by_{s}_{v}_norm'
            LAYER_DATA_NAME = f'While_Conv_with_S_{CORR_TYPE}'
            try:
                with open(PATH + LAYER_DATA_NAME + '.pkl', 'rb') as fr:
                        res = pickle.load(fr)
            except:
                continue

            xs, m, s = [], [], []
            a = compute_avg(res)
            m.append(a[0])
            s.append(a[1])
            RES[LAYER_DATA_NAME.replace(str(v), '_')].append((np.nanmean(m), np.nanmean(s)))
            
r = {}
for k, v in RES.items():
    k_ = k.replace('While_Conv_with_S_', '').replace('___norm', '').replace('_', ' ')
    k_ = k_[0].upper() + k_[1:]
    
    r[k_] = ([round(v_[0], 5) for v_ in v], [round(v_[1], 5) for v_ in v])

print(r)